# Neo4j + Vercel AI SDK Integration Demo

Three patterns for integrating Neo4j with the [Vercel AI SDK](https://sdk.vercel.ai/):
1. **Direct Neo4j queries** — explore the schema and run Cypher
2. **AI agent with Neo4j tools** — natural-language queries via `generateText` + `tool()`
3. **Persistent agent memory** — cross-session memory via `neo4j-agent-memory`

The demo uses the publicly accessible **Neo4j companies knowledge graph** (`Organization` nodes linked to `Article` nodes via `[:MENTIONS]` relationships).

## 1. Setup

Add Node.js to PATH and install npm packages.

In [ ]:
import os, subprocess

# Add Node.js 20 to PATH (installed at ~/node20/)
node_bin = os.path.expanduser('~/node20/bin')
os.environ['PATH'] = node_bin + ':' + os.environ.get('PATH', '')

r = subprocess.run(['node', '--version'], capture_output=True, text=True)
print('Node.js:', r.stdout.strip() or r.stderr.strip())

pkg_dir = os.getcwd()  # notebook kernel CWD = vercel-agent/ directory
if not os.path.isdir(os.path.join(pkg_dir, 'node_modules', 'ai')):
    print('Installing npm packages...')
    r2 = subprocess.run(
        ['npm', 'install', 'ai', '@ai-sdk/openai', 'neo4j-driver', 'neo4j-agent-memory', 'zod@3'],
        cwd=pkg_dir, env=os.environ, capture_output=True, text=True
    )
    print(r2.stdout[-500:] if r2.stdout else r2.stderr[-500:])
else:
    print('npm packages already installed ✓')

## 2. Configuration

Set environment variables for Neo4j and OpenAI.

- The **companies demo database** is used for knowledge-graph queries (read-only, public).
- **Agent memory** requires a separate **writable** Neo4j instance (Neo4j AuraFree, local Docker, etc.).

In [ ]:
import os

# Neo4j Knowledge Graph (read-only companies demo)
os.environ['NEO4J_URI']      = 'neo4j+s://demo.neo4jlabs.com:7687'
os.environ['NEO4J_USERNAME'] = 'companies'
os.environ['NEO4J_PASSWORD'] = 'companies'
os.environ['NEO4J_DATABASE'] = 'companies'

# Neo4j Memory DB (requires WRITE access)
# Leave as placeholder to see graceful warning in memory cells.
# Replace with a real writable instance (AuraFree, local Docker, etc.)
os.environ.setdefault('NEO4J_MEMORY_URI',      'bolt+s://your-memory-instance.databases.neo4j.io:7687')
os.environ.setdefault('NEO4J_MEMORY_USERNAME',  'neo4j')
os.environ.setdefault('NEO4J_MEMORY_PASSWORD',  'your-aura-password')
os.environ.setdefault('NEO4J_MEMORY_DATABASE',  'neo4j')

# OpenAI API Key
os.environ.setdefault('OPENAI_API_KEY', 'sk-your-openai-api-key')

print('Environment configured.')
print('Neo4j URI:   ', os.environ['NEO4J_URI'])
print('Memory URI:  ', os.environ['NEO4J_MEMORY_URI'])
print('OpenAI key:  ', os.environ['OPENAI_API_KEY'][:8] + '...')

## 3. Direct Neo4j Query (no AI)

Verify connectivity and explore the companies graph schema.

In [ ]:
script = r"""
import neo4j from 'neo4j-driver';

const driver = neo4j.driver(
  process.env.NEO4J_URI,
  neo4j.auth.basic(process.env.NEO4J_USERNAME, process.env.NEO4J_PASSWORD)
);
const db = process.env.NEO4J_DATABASE;

const { records } = await driver.executeQuery(
  `MATCH (a:Article)-[:MENTIONS]->(o:Organization)
   RETURN o.name AS company, COUNT(a) AS articles
   ORDER BY articles DESC LIMIT 10`,
  {},
  { database: db }
);

console.log('Top 10 Organizations by News Coverage:\n');
console.log('Company'.padEnd(45) + 'Articles');
console.log('-'.repeat(55));
records.forEach(r => {
  const name     = (r.get('company') ?? 'N/A').toString().slice(0, 43).padEnd(45);
  const articles = r.get('articles').toString();
  console.log(`${name}${articles}`);
});

const { records: labelRecs } = await driver.executeQuery('CALL db.labels()', {}, { database: db });
console.log('\nAvailable node labels:', labelRecs.map(r => r.get('label')).join(', '));

const { records: sample } = await driver.executeQuery(
  'MATCH (o:Organization) RETURN o LIMIT 1', {}, { database: db }
);
if (sample.length) {
  const props = Object.keys(sample[0].get('o').properties).join(', ');
  const name  = sample[0].get('o').properties.name;
  const summ  = (sample[0].get('o').properties.summary ?? '').slice(0, 80);
  console.log('Organization properties:', props);
  console.log('Example:', name, '|', summ);
}

await driver.close();
"""

dest = os.path.join(pkg_dir, 'neo4j_direct.mjs')
with open(dest, 'w') as f:
    f.write(script.strip())
print('Written', dest)

In [ ]:
import subprocess, os
result = subprocess.run(
    ['node', 'neo4j_direct.mjs'],
    cwd=pkg_dir,
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print('STDERR:', result.stderr[:500])

## 4. AI Agent with Neo4j Tools

Use `generateText` + `tool()` to create an agent that queries Neo4j on demand.

> **API note (v6):** `tool()` uses `inputSchema: jsonSchema({...})` — not `parameters: z.object(...)`.

In [ ]:
script = r"""
import { generateText, tool, jsonSchema } from 'ai';
import { openai } from '@ai-sdk/openai';
import neo4j from 'neo4j-driver';

const driver = neo4j.driver(
  process.env.NEO4J_URI,
  neo4j.auth.basic(process.env.NEO4J_USERNAME, process.env.NEO4J_PASSWORD)
);

// AI SDK v6: use inputSchema with jsonSchema() helper (not parameters: z.object(...))
const searchCompanies = tool({
  description: 'Search for organizations in Neo4j by name keyword',
  inputSchema: jsonSchema({
    type: 'object',
    properties: {
      keyword: { type: 'string', description: 'Organization name or topic keyword' },
      limit:   { type: 'number', description: 'Max results', default: 5 },
    },
    required: ['keyword'],
  }),
  execute: async ({ keyword, limit = 5 }) => {
    const { records } = await driver.executeQuery(
      `MATCH (o:Organization)
       WHERE toLower(o.name)    CONTAINS toLower($keyword)
          OR toLower(o.summary) CONTAINS toLower($keyword)
       RETURN o.name AS name, o.summary AS summary, o.nbrEmployees AS employees
       LIMIT $limit`,
      { keyword, limit },
      { database: process.env.NEO4J_DATABASE }
    );
    return records.map(r => ({ name: r.get('name'), summary: r.get('summary'), employees: r.get('employees') }));
  },
});

const getCompanyArticles = tool({
  description: 'Get recent news articles mentioning a specific organization',
  inputSchema: jsonSchema({
    type: 'object',
    properties: {
      companyName: { type: 'string', description: 'Exact organization name' },
      limit:       { type: 'number', description: 'Max articles', default: 3 },
    },
    required: ['companyName'],
  }),
  execute: async ({ companyName, limit = 3 }) => {
    const { records } = await driver.executeQuery(
      `MATCH (a:Article)-[:MENTIONS]->(o:Organization)
       WHERE o.name = $companyName
       RETURN a.title AS title, a.sentiment AS sentiment, a.summary AS summary, a.date AS date
       ORDER BY a.date DESC LIMIT $limit`,
      { companyName, limit },
      { database: process.env.NEO4J_DATABASE }
    );
    return records.map(r => ({ title: r.get('title'), sentiment: r.get('sentiment'), summary: r.get('summary'), date: r.get('date') }));
  },
});

console.log('Running agent with Neo4j tools...\n');

const { text, steps } = await generateText({
  model:    openai.chat('gpt-5.4'),
  system:   'You are a business research analyst with access to a Neo4j organization knowledge graph.',
  prompt:   'Find 3 technology-related organizations and for the most prominent one, retrieve its recent news coverage. Provide a short summary.',
  tools:    { searchCompanies, getCompanyArticles },
  stopWhen: stepCountIs(10),
});

console.log('=== Agent Response ===');
console.log(text);
console.log(`\n[Completed in ${steps.length} step(s)]`);
steps.forEach((step, i) => {
  const toolCalls = step.toolCalls?.map(tc => tc.toolName).join(', ') || 'none';
  console.log(`  Step ${i + 1}: [${toolCalls}]`);
});

await driver.close();
"""

dest = os.path.join(pkg_dir, 'agent_tools.mjs')
with open(dest, 'w') as f:
    f.write(script.strip())
print('Written', dest)

In [ ]:
import subprocess, os
result = subprocess.run(
    ['node', 'agent_tools.mjs'],
    cwd=pkg_dir,
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print('STDERR:', result.stderr[:500])

## 5. Persistent Agent Memory with `neo4j-agent-memory`

[`neo4j-agent-memory`](https://www.npmjs.com/package/neo4j-agent-memory) stores agent knowledge as a graph in Neo4j, enabling cross-session persistence via hybrid retrieval.

> **Prerequisite:** Memory requires a **writable** Neo4j instance. Set `NEO4J_MEMORY_*` env vars in cell 2.
> 
> **Protocol note:** `neo4j-agent-memory` bundles `neo4j-driver` v5 which does not support `neo4j+s://` routing. Use `bolt+s://` — handled automatically in the script below.

In [ ]:
script = r"""
import { generateText } from 'ai';
import { openai } from '@ai-sdk/openai';
import { Neo4jAgentMemory } from 'neo4j-agent-memory';

const memoryUri      = process.env.NEO4J_MEMORY_URI      || process.env.NEO4J_URI;
const memoryUser     = process.env.NEO4J_MEMORY_USERNAME  || process.env.NEO4J_USERNAME;
const memoryPassword = process.env.NEO4J_MEMORY_PASSWORD  || process.env.NEO4J_PASSWORD;
const memoryDatabase = process.env.NEO4J_MEMORY_DATABASE  || process.env.NEO4J_DATABASE;

// neo4j-agent-memory bundles driver v5 which doesn't support neo4j+s:// routing
const boltUri = memoryUri.replace(/^neo4j\+s:\/\//, 'bolt+s://');

if (!process.env.NEO4J_MEMORY_URI) {
  console.warn('WARNING: NEO4J_MEMORY_URI not set.');
  console.warn('  Memory requires a writable Neo4j instance (AuraFree etc.).\n');
}

let memory;
try {
  memory = new Neo4jAgentMemory({
    uri:      boltUri,
    username: memoryUser,
    password: memoryPassword,
    database: memoryDatabase,
    agentId:  'vercel-demo-agent',
  });
  await memory.initialize();
  console.log('Neo4j memory initialized\n');
} catch (err) {
  console.error('Memory initialization failed:', err.message);
  console.error('  -> Set NEO4J_MEMORY_* vars to a writable Neo4j instance.');
  process.exit(0);
}

async function buildSystemPrompt() {
  const memories = await memory.loadMemories();
  const memText  = memories.length
    ? '\n\nFrom previous sessions:\n' + memories.map(m => `- ${m.content}`).join('\n')
    : '';
  return `You are a helpful research assistant with persistent memory backed by Neo4j.${memText}`;
}

// Turn 1
const system1 = await buildSystemPrompt();
const { text: response1 } = await generateText({
  model:  openai.chat('gpt-5.4'),
  system: system1,
  prompt: "My name is Alex. I'm researching supply chain risks for retail companies.",
});
console.log('Turn 1:', response1);
await memory.saveMemory("User is Alex. Researching supply chain risks for retail companies.");

// Turn 2 - demonstrates memory persistence
const system2 = await buildSystemPrompt();
const { text: response2 } = await generateText({
  model:  openai.chat('gpt-5.4'),
  system: system2,
  prompt: "What do you remember about me and my research focus?",
});
console.log('\nTurn 2:', response2);

await memory.close();
console.log('\nMemory session complete.');
"""

dest = os.path.join(pkg_dir, 'memory_agent.mjs')
with open(dest, 'w') as f:
    f.write(script.strip())
print('Written', dest)

In [ ]:
import subprocess, os
result = subprocess.run(
    ['node', 'memory_agent.mjs'],
    cwd=pkg_dir,
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print('STDERR:', result.stderr[:500])

## 6. Full Industry Research Agent

Combines all three patterns:
- **Neo4j Cypher tools** — search organizations, fetch news articles, find co-mentioned peers
- **`neo4j-agent-memory`** — retrieve prior context and store new insights
- **Multi-step agentic loop** — autonomous tool selection via `stopWhen: stepCountIs(N)` (AI SDK v6+)

In [ ]:
script = r"""
import { generateText, tool, jsonSchema } from 'ai';
import { openai } from '@ai-sdk/openai';
import neo4j from 'neo4j-driver';
import { Neo4jAgentMemory } from 'neo4j-agent-memory';

const driver = neo4j.driver(
  process.env.NEO4J_URI,
  neo4j.auth.basic(process.env.NEO4J_USERNAME, process.env.NEO4J_PASSWORD)
);

const memoryUri  = (process.env.NEO4J_MEMORY_URI || process.env.NEO4J_URI).replace(/^neo4j\+s:\/\//, 'bolt+s://');
const memoryUser = process.env.NEO4J_MEMORY_USERNAME || process.env.NEO4J_USERNAME;
const memoryPass = process.env.NEO4J_MEMORY_PASSWORD || process.env.NEO4J_PASSWORD;
const memoryDb   = process.env.NEO4J_MEMORY_DATABASE || process.env.NEO4J_DATABASE;

const searchCompanies = tool({
  description: 'Search organizations in Neo4j by keyword',
  inputSchema: jsonSchema({
    type: 'object',
    properties: {
      keyword: { type: 'string' },
      limit:   { type: 'number', default: 5 },
    },
    required: ['keyword'],
  }),
  execute: async ({ keyword, limit = 5 }) => {
    const { records } = await driver.executeQuery(
      `MATCH (o:Organization)
       WHERE toLower(o.name) CONTAINS toLower($keyword)
          OR toLower(o.summary) CONTAINS toLower($keyword)
       RETURN o.name AS name, o.summary AS summary, o.nbrEmployees AS employees
       LIMIT $limit`,
      { keyword, limit },
      { database: process.env.NEO4J_DATABASE }
    );
    return records.map(r => ({ name: r.get('name'), summary: r.get('summary'), employees: r.get('employees') }));
  },
});

const getCompanyArticles = tool({
  description: 'Get recent news articles mentioning a specific organization',
  inputSchema: jsonSchema({
    type: 'object',
    properties: {
      companyName: { type: 'string' },
      limit:       { type: 'number', default: 3 },
    },
    required: ['companyName'],
  }),
  execute: async ({ companyName, limit = 3 }) => {
    const { records } = await driver.executeQuery(
      `MATCH (a:Article)-[:MENTIONS]->(o:Organization)
       WHERE o.name = $companyName
       RETURN a.title AS title, a.sentiment AS sentiment, a.summary AS summary
       ORDER BY a.date DESC LIMIT $limit`,
      { companyName, limit },
      { database: process.env.NEO4J_DATABASE }
    );
    return records.map(r => ({ title: r.get('title'), sentiment: r.get('sentiment'), summary: r.get('summary') }));
  },
});

const getRelatedCompanies = tool({
  description: 'Find organizations frequently co-mentioned with a given org',
  inputSchema: jsonSchema({
    type: 'object',
    properties: {
      companyName: { type: 'string' },
      limit:       { type: 'number', default: 5 },
    },
    required: ['companyName'],
  }),
  execute: async ({ companyName, limit = 5 }) => {
    const { records } = await driver.executeQuery(
      `MATCH (a:Article)-[:MENTIONS]->(src:Organization { name: $companyName })
       MATCH (a)-[:MENTIONS]->(related:Organization)
       WHERE related.name <> $companyName
       RETURN related.name AS name, count(a) AS sharedArticles
       ORDER BY sharedArticles DESC LIMIT $limit`,
      { companyName, limit },
      { database: process.env.NEO4J_DATABASE }
    );
    return records.map(r => ({ name: r.get('name'), sharedArticles: r.get('sharedArticles').toNumber() }));
  },
});

let memory;
try {
  memory = new Neo4jAgentMemory({ uri: memoryUri, username: memoryUser, password: memoryPass, database: memoryDb, agentId: 'research-agent' });
  await memory.initialize();
  console.log('Memory initialized\n');
} catch (err) {
  console.warn(`Memory unavailable (${err.message}) – continuing without persistence.\n`);
  memory = null;
}

const pastMemories = memory ? await memory.loadMemories() : [];
const memContext   = pastMemories.length
  ? '\n\nFrom previous sessions:\n' + pastMemories.map(m => `- ${m.content}`).join('\n')
  : '';

const { text, steps } = await generateText({
  model:    openai.chat('gpt-5.4'),
  system:   `You are an industry research analyst with access to a Neo4j knowledge graph. Use tools to provide data-backed answers.${memContext}`,
  prompt:   'Research the top technology companies in the graph. For the most prominent one, find its recent news and frequently co-mentioned peers. Provide a structured research brief.',
  tools:    { searchCompanies, getCompanyArticles, getRelatedCompanies },
  stopWhen: stepCountIs(10),
});

console.log('=== Research Brief ===\n');
console.log(text);
console.log(`\n[${steps.length} step(s)]`);
steps.forEach((s, i) => console.log(`  Step ${i+1}: [${s.toolCalls?.map(tc => tc.toolName).join(', ') || 'none'}]`));

if (memory) {
  await memory.saveMemory('Researched top technology companies: news and co-mention analysis.');
  await memory.close();
}
await driver.close();
"""

dest = os.path.join(pkg_dir, 'research_agent.mjs')
with open(dest, 'w') as f:
    f.write(script.strip())
print('Written', dest)

In [ ]:
import subprocess, os
result = subprocess.run(
    ['node', 'research_agent.mjs'],
    cwd=pkg_dir,
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print('STDERR:', result.stderr[:500])

## 7. Summary

| Pattern | Key APIs | Use Case |
|---------|----------|----------|
| Direct Query | `neo4j-driver` | Schema exploration, reporting |
| Agent + Tools | `generateText`, `tool()`, `jsonSchema()` | Dynamic graph queries via natural language |
| Agent + Memory | `Neo4jAgentMemory` | Persistent, cross-session knowledge |

### Key Implementation Notes

- **AI SDK v6:** Use `inputSchema: jsonSchema({...})` in `tool()` — not `parameters: z.object(...)`.
- **`openai.chat('gpt-5.4')`** forces Chat Completions API.
- **`bolt+s://`** required for `neo4j-agent-memory` (bundles driver v5, no `neo4j+s://` routing).
- **Separate memory DB:** `neo4j-agent-memory` needs write access — keep it separate from read-only knowledge graphs.

### Next Steps

- Deploy to **Vercel Edge Functions** / **Next.js API routes** using the same `generateText` pattern.
- Use `@neo4j/mcp-server` to expose the knowledge graph to any MCP-compatible client.
- Explore `streamText` for streaming responses in chat UIs.